# De una pregunta física a una consulta reproducible
## Introducción práctica a Python y bases de datos

**Versión para estudiantes** · **Duración:** 2 horas · **Entorno:** Google Colab · **Nivel:** inicial

Las celdas marcadas con `TODO` son los espacios de trabajo. Predice el resultado antes de ejecutar y conserva tus intentos, incluso cuando aparezcan errores.

Trabajaremos con datos **sintéticos** inspirados en eventos de física de altas energías. No representan resultados reales de ningún experimento.

Al final podrás usar Python para explorar datos y SQL para seleccionar, resumir y relacionar tablas.

## 0. Preparar Colab

Guarda tu copia: **Archivo → Guardar una copia en Drive**. Ejecuta una celda con el botón ▶ o con `Shift + Enter`.

In [ ]:
print("¡Hola, física de altas energías!")

## 1. Python esencial

Una variable asocia un nombre con un valor. Python distingue entre números, texto y valores lógicos.

In [ ]:
energia_gev = 125.4
numero_muones = 2
calidad = "buena"
pasa_seleccion = energia_gev > 50 and numero_muones >= 2

print(type(energia_gev), type(numero_muones), type(calidad))
print("¿Pasa la selección?", pasa_seleccion)

### Listas, ciclos y condicionales

Una lista agrupa valores. Un ciclo repite una acción y un condicional decide qué camino seguir. Antes de ejecutar: ¿cuántas energías crees que se imprimirán?

In [ ]:
energias = [12.5, 48.0, 81.2, 125.4, 37.8]

for energia in energias:
    if energia > 50:
        print(energia, "GeV: candidata")
    else:
        print(energia, "GeV: no pasa el umbral")

### Funciones

Una función da nombre a una regla y permite reutilizarla.

In [ ]:
def energia_en_tev(energia_gev):
    return energia_gev / 1000

print(energia_en_tev(125.4), "TeV")

### Reto 1 — Construir una regla de selección (8 min)

Completa `es_candidato`. Debe regresar `True` únicamente si:

- la energía es **mayor que** 50 GeV;
- hay al menos 2 muones;
- la calidad es `"buena"`.

Luego prueba los tres eventos. Antes de ejecutar, anota cuál debería pasar.

In [ ]:
def es_candidato(energia_gev, numero_muones, calidad):
    # TODO: reemplaza la siguiente línea con tu condición
    return False

eventos_prueba = [
    {"evento_id": "E-101", "energia_gev": 50, "numero_muones": 2, "calidad": "buena"},
    {"evento_id": "E-102", "energia_gev": 91, "numero_muones": 2, "calidad": "buena"},
    {"evento_id": "E-103", "energia_gev": 140, "numero_muones": 1, "calidad": "buena"},
]

for evento in eventos_prueba:
    resultado = es_candidato(evento["energia_gev"], evento["numero_muones"], evento["calidad"])
    print(evento["evento_id"], resultado)

## 2. Datos tabulares con pandas

Crearemos 60 eventos reproducibles. Una fila representa un evento y cada columna, una propiedad. Los datos son didácticos y sintéticos.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 60
eventos = pd.DataFrame({
    "evento_id": range(1001, 1001 + n),
    "detector_id": rng.choice([1, 2, 3], size=n, p=[0.4, 0.35, 0.25]),
    "energia_gev": np.round(rng.gamma(shape=2.2, scale=38, size=n), 2),
    "numero_trazas": rng.integers(2, 80, size=n),
    "numero_muones": rng.choice([0, 1, 2, 3, 4], size=n, p=[0.28, 0.30, 0.24, 0.13, 0.05]),
    "calidad": rng.choice(["buena", "revisar"], size=n, p=[0.82, 0.18]),
})
eventos.head()

In [ ]:
print("Forma (filas, columnas):", eventos.shape)
print("Columnas:", list(eventos.columns))
eventos[["energia_gev", "numero_trazas", "numero_muones"]].describe().round(2)

### Reto 2 — Filtrar una tabla (7 min)

Crea `seleccion` con eventos de energía mayor a 80 GeV, al menos 2 muones y calidad buena. En pandas, cada condición va entre paréntesis y se combina con `&`.

In [ ]:
# TODO: completa las tres condiciones
seleccion = eventos[(eventos["energia_gev"] > 80)]

print("Eventos seleccionados:", len(seleccion))
seleccion.head()

## 3. ¿Por qué una base de datos?

Un `DataFrame` es una tabla en memoria. Una base de datos organiza tablas relacionadas y permite consultarlas sin cargar necesariamente todo.

- **Llave primaria:** identifica una fila de manera única.
- **Llave foránea:** apunta a una fila de otra tabla.
- **SQL:** lenguaje para preguntar y modificar datos estructurados.

Usaremos SQLite: una base completa guardada en un solo archivo y disponible en Python sin instalar nada.

In [ ]:
import sqlite3

detectores = pd.DataFrame({
    "detector_id": [1, 2, 3],
    "nombre": ["Aurora", "Quark", "Bosón"],
    "ubicacion": ["Punto A", "Punto B", "Punto C"],
})

conexion = sqlite3.connect("eventos_clase.db")
detectores.to_sql("detectores", conexion, if_exists="replace", index=False)
eventos.to_sql("eventos", conexion, if_exists="replace", index=False)
print("Base creada. Tablas: detectores, eventos")

### SELECT, WHERE y ORDER BY

Lee la consulta en tres preguntas: ¿qué columnas?, ¿de qué tabla?, ¿qué filas?

In [ ]:
consulta = """
SELECT evento_id, energia_gev, numero_muones
FROM eventos
WHERE energia_gev > 80 AND calidad = 'buena'
ORDER BY energia_gev DESC
LIMIT 10;
"""

pd.read_sql_query(consulta, conexion)

### Consultas con parámetros

El signo `?` reserva el lugar para un valor. No construyas consultas concatenando texto ingresado por una persona.

In [ ]:
umbral = 100
consulta_parametrizada = """
SELECT evento_id, energia_gev, calidad
FROM eventos
WHERE energia_gev > ?
ORDER BY energia_gev DESC;
"""
pd.read_sql_query(consulta_parametrizada, conexion, params=(umbral,)).head()

### Reto 3 — Escribir una consulta (6 min)

Muestra `evento_id`, `detector_id`, `energia_gev` y `numero_muones` para eventos con energía mayor a 100 GeV y al menos 2 muones. Ordena de mayor a menor energía.

In [ ]:
reto_3 = """
-- TODO: escribe aquí tu SELECT
"""
# Quita el # de la siguiente línea cuando tu consulta esté lista.
# pd.read_sql_query(reto_3, conexion)

### GROUP BY: resumir grupos

`COUNT`, `AVG`, `MIN`, `MAX` y `SUM` condensan muchas filas.

In [ ]:
resumen = """
SELECT detector_id,
       COUNT(*) AS numero_eventos,
       ROUND(AVG(energia_gev), 2) AS energia_promedio_gev,
       MAX(energia_gev) AS energia_maxima_gev
FROM eventos
GROUP BY detector_id
ORDER BY energia_promedio_gev DESC;
"""
pd.read_sql_query(resumen, conexion)

### JOIN: relacionar tablas

Los eventos guardan `detector_id`; la tabla `detectores` traduce ese identificador en información legible.

In [ ]:
consulta_join = """
SELECT e.evento_id, d.nombre AS detector, e.energia_gev, e.numero_muones
FROM eventos AS e
JOIN detectores AS d ON e.detector_id = d.detector_id
WHERE e.calidad = 'buena'
ORDER BY e.energia_gev DESC
LIMIT 10;
"""
pd.read_sql_query(consulta_join, conexion)

### Manejo básico: insertar, actualizar y confirmar

En una base real, valida permisos y conserva un respaldo. Aquí añadimos un evento didáctico, lo actualizamos y confirmamos la transacción.

In [ ]:
cursor = conexion.cursor()
cursor.execute(
    "INSERT INTO eventos VALUES (?, ?, ?, ?, ?, ?)",
    (9999, 1, 110.5, 24, 2, "revisar")
)
cursor.execute(
    "UPDATE eventos SET calidad = ? WHERE evento_id = ?",
    ("buena", 9999)
)
conexion.commit()
pd.read_sql_query("SELECT * FROM eventos WHERE evento_id = ?", conexion, params=(9999,))

## 4. Reto integrador en parejas (16 min)

Elige **una** pregunta:

1. ¿Qué detector tiene mayor energía promedio entre eventos de buena calidad?
2. ¿Cuántos eventos superan 100 GeV en cada detector?
3. ¿Qué porcentaje de eventos contiene al menos dos muones, por detector?

Entrega cuatro elementos: consulta SQL, resultado, gráfica y una frase de interpretación. Añade una limitación: recuerda que estos datos son sintéticos y no demuestran un fenómeno físico.

In [ ]:
pregunta = "Escribe aquí la pregunta elegida"

consulta_final = """
-- TODO: escribe tu consulta. Probablemente necesitarás JOIN y GROUP BY.
"""

# resultado = pd.read_sql_query(consulta_final, conexion)
# resultado

In [ ]:
# Adapta los nombres de las columnas a tu resultado.
# resultado.plot.bar(x="nombre", y=resultado.columns[-1], legend=False,
#                    title=pregunta, xlabel="Detector", ylabel="Resultado");

**Nuestro hallazgo:** ...  
**Una limitación:** ...

## 5. Cierre

Hoy recorriste un flujo completo:

**pregunta → datos → consulta → resultado → interpretación**

Ticket de salida:

1. Copia una consulta que te haya funcionado.
2. Explica su resultado en una oración.
3. Escribe un concepto que quieras practicar otra vez.

> Una consulta correcta no garantiza una conclusión física correcta. También importan la calibración, las incertidumbres, los sesgos y el significado de las variables.

In [ ]:
conexion.close()
print("Conexión cerrada. ¡Buen trabajo!")